# 01 — Data Exploration
PS69 Weather Analytics | Phase 1 baseline dataset: ERA5 reanalysis, Jabalpur (23.25°N, 80.0°E), hourly, 2024-01-01 to 2025-12-31.

**Goal of this notebook:** inspect the raw file, confirm structure, and understand exactly what we have before doing anything else.

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
from data.load_clean import load_raw

pd.set_option('display.max_columns', None)

## Note on the raw file
The file downloaded from Copernicus CDS is a **ZIP archive saved with a `.csv` extension**
(this is normal for CDS API downloads). `load_raw()` detects this automatically via the
file's magic bytes and extracts the real CSV in memory.

In [2]:
df = load_raw('../data/raw/jabalpur_weather_2024_2025.csv')
print('Shape:', df.shape)
df.head()

Shape: (17544, 10)


,valid_time,u10,v10,fg10,d2m,t2m,msl,tp,latitude,longitude
0,2024-01-01 00:00:00,-1.761932,-0.232742,2.527000,284.76923,286.20460,101666.750,0.0,23.25,80.0
1,2024-01-01 01:00:00,-1.806198,0.186417,2.771432,284.72327,286.03530,101697.560,0.0,23.25,80.0
2,2024-01-01 02:00:00,-1.589249,0.230621,2.776984,285.11620,286.82098,101775.125,0.0,23.25,80.0
3,2024-01-01 03:00:00,-1.512772,0.407822,3.739992,285.19455,287.47992,101832.560,0.0,23.25,80.0
4,2024-01-01 04:00:00,-1.355469,0.831421,4.413458,285.62860,288.27643,101880.060,0.0,23.25,80.0


In [3]:
print('Columns:', list(df.columns))
print()
print(df.dtypes)

Columns: ['valid_time', 'u10', 'v10', 'fg10', 'd2m', 't2m', 'msl', 'tp', 'latitude', 'longitude']

valid_time        str
u10           float64
v10           float64
fg10          float64
d2m           float64
t2m           float64
msl           float64
tp            float64
latitude      float64
longitude     float64
dtype: object


## Column meanings (ERA5 single-levels variables)
| Column | Meaning | Native unit |
|---|---|---|
| valid_time | Timestamp (UTC) | datetime |
| u10 | 10m wind, east-west component | m/s |
| v10 | 10m wind, north-south component | m/s |
| fg10 | 10m wind gust | m/s |
| d2m | 2m dewpoint temperature | Kelvin |
| t2m | 2m air temperature | Kelvin |
| msl | Mean sea level pressure | Pascals |
| tp | Total precipitation (accumulated over the hour) | metres |
| latitude / longitude | Grid point location | degrees |

Note: `fg10` (wind gust) is present in addition to the 6 variables listed in the brief — one extra usable feature.

In [4]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate timestamps:', df.duplicated(subset=['valid_time']).sum())

Missing values per column:
valid_time    0
u10           0
v10           0
fg10          0
d2m           0
t2m           0
msl           0
tp            0
latitude      0
longitude     0
dtype: int64

Duplicate rows: 0
Duplicate timestamps: 0


In [5]:
df['valid_time'] = pd.to_datetime(df['valid_time'])
print('Date range:', df['valid_time'].min(), 'to', df['valid_time'].max())

diffs = df['valid_time'].diff().dropna()
print()
print('Sampling frequency (value counts of time deltas):')
print(diffs.value_counts())

Date range: 2024-01-01 00:00:00 to 2025-12-31 23:00:00

Sampling frequency (value counts of time deltas):
valid_time
0 days 01:00:00    17543
Name: count, dtype: int64


In [6]:
expected = pd.date_range(df['valid_time'].min(), df['valid_time'].max(), freq='h')
missing_times = set(expected) - set(df['valid_time'])
print('Expected hourly rows:', len(expected))
print('Actual rows:', len(df))
print('Missing timestamps:', len(missing_times))

Expected hourly rows: 17544
Actual rows: 17544
Missing timestamps: 0


In [7]:
print('Unique lat/lon points:')
print(df[['latitude','longitude']].drop_duplicates())

Unique lat/lon points:
   latitude  longitude
0     23.25       80.0


**This is a single grid-point time series, not spatial data.** Any conclusions from this
Phase-1 baseline are specific to this one location — this is an explicit limitation to state
in the SIH report.

In [8]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
valid_time,17544,2024-12-31 11:30:00,2024-01-01 00:00:00,2024-07-01 17:45:00,2024-12-31 11:30:00,2025-07-02 05:15:00,2025-12-31 23:00:00,NaN
u10,17544.0,0.421467,-4.639572,-1.147572,0.212456,1.7775,7.896408,1.898894
v10,17544.0,-0.215067,-6.516525,-1.484009,-0.413887,0.912933,6.945709,1.656045
fg10,17544.0,5.012513,0.624704,3.350292,4.56843,6.382217,16.018845,2.229384
d2m,17544.0,290.72187,274.11276,285.471405,290.78781,296.966837,300.72754,6.031958
t2m,17544.0,298.748565,280.19217,295.2209,299.22458,302.711725,317.06488,6.540125
msl,17544.0,100879.458146,99224.81,100337.375,100885.3425,101433.89125,102289.94,646.044966
tp,17544.0,0.000157,0.0,0.0,0.0,0.000015,0.020897,0.000664
latitude,17544.0,23.25,23.25,23.25,23.25,23.25,23.25,0.0
longitude,17544.0,80.0,80.0,80.0,80.0,80.0,80.0,0.0


## Summary of Notebook 01
- 17,544 rows, 10 columns, perfectly continuous hourly data, zero missing values, zero duplicates
- Single grid point (Jabalpur), 2 full years (2024-2025)
- Units are ERA5-native (Kelvin, Pascals, metres) — converted in the next notebook
- Next: `02_data_quality_cleaning.ipynb` — formal quality checks and unit conversion